In [ ]:
import json
import time
import traceback
from pathlib import Path

import requests
from cryptography.fernet import Fernet

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def _find_first(filename: str) -> Path | None:
    for path in INPUT_ROOT.rglob(filename):
        if path.is_file():
            return path
    return None


def load_config() -> dict:
    config_path = _find_first("config.json")
    if config_path is None:
        raise RuntimeError("config.json not found under /kaggle/input")
    return json.loads(config_path.read_text(encoding="utf-8"))


def load_secrets() -> dict:
    enc_path = _find_first("secrets.enc")
    key_path = _find_first("fernet.key")
    if enc_path is None or key_path is None:
        raise RuntimeError("secrets.enc/fernet.key not found under /kaggle/input")
    payload = Fernet(key_path.read_bytes().strip()).decrypt(enc_path.read_bytes())
    return json.loads(payload.decode("utf-8"))


def pick_secret(config: dict, secrets: dict) -> tuple[str, str]:
    names = [str(config.get("secret_env_var") or "").strip()]
    names.extend(str(item).strip() for item in (config.get("secret_env_aliases") or []))
    for name in names:
        if name and str(secrets.get(name) or "").strip():
            return name, str(secrets[name]).strip()
    raise RuntimeError(f"Secret not found in bundle for lookup order: {names}")


def extract_response_text(payload: dict) -> str:
    texts: list[str] = []
    for candidate in payload.get("candidates") or []:
        content = candidate.get("content") or {}
        for part in content.get("parts") or []:
            text = str(part.get("text") or "").strip()
            if text:
                texts.append(text)
    return "\n".join(texts).strip()


In [ ]:
# This notebook is permanently retired because it bypassed the cross-process
# Google AI quota ledger. Use a GoogleAIClient-based probe with the dedicated
# GOOGLE_AI_LIMITER_SUPABASE_* configuration instead. No config flag may
# re-enable this raw-key transport.
raise RuntimeError(
    "GemmaKey2Probe retired: direct Google provider calls are prohibited; "
    "use GoogleAIClient with shared limiter accounting"
)
